# Circuit Builders

qdk-pythonic provides factory functions for common quantum circuits and states.
These save you from manually wiring up well-known patterns every time.

In [ ]:
from qdk_pythonic import bell_state, ghz_state, w_state, qft, inverse_qft, random_circuit

## Bell State

The simplest entangled circuit: produces (|00> + |11>) / sqrt(2).

In [ ]:
circ = bell_state()
print(circ.draw())
print(f"Qubits: {circ.qubit_count()}, Depth: {circ.depth()}, Gates: {circ.gate_count()}")

Pass `measure=True` to append measurements:

In [ ]:
measured = bell_state(measure=True)
print(measured.to_qsharp())

## GHZ State

Generalizes the Bell state to N qubits: (|00...0> + |11...1>) / sqrt(2).

In [ ]:
ghz5 = ghz_state(5)
print(ghz5.draw())
print(f"Qubits: {ghz5.qubit_count()}, Depth: {ghz5.depth()}, Gates: {ghz5.gate_count()}")

In [ ]:
print(ghz_state(5).to_openqasm())

## W State

The W state distributes a single excitation evenly across N qubits:
(|10...0> + |01...0> + ... + |00...1>) / sqrt(n).

Unlike GHZ, the W state retains entanglement even if one qubit is lost.

In [ ]:
w3 = w_state(3)
print(w3.draw())
print(f"Qubits: {w3.qubit_count()}, Depth: {w3.depth()}, Gates: {w3.gate_count()}")

In [ ]:
print(w_state(4).to_qsharp())

## Quantum Fourier Transform (QFT)

The QFT is a key subroutine in many quantum algorithms (Shor's, phase estimation).
It uses Hadamard gates and controlled phase rotations.

In [ ]:
qft3 = qft(3)
print(qft3.draw())
print(f"Qubits: {qft3.qubit_count()}, Depth: {qft3.depth()}, Gates: {qft3.gate_count()}")

In [ ]:
print("=== Q# ===")
print(qft(3).to_qsharp())
print()
print("=== OpenQASM 3.0 ===")
print(qft(3).to_openqasm())

## Inverse QFT

The adjoint of the QFT, used to decode results in phase estimation and similar algorithms.

In [ ]:
iqft3 = inverse_qft(3)
print(iqft3.draw())
print(f"Qubits: {iqft3.qubit_count()}, Depth: {iqft3.depth()}, Gates: {iqft3.gate_count()}")

## Random Circuit

Generate random circuits for benchmarking. Use `seed` for reproducibility.

In [ ]:
rc = random_circuit(4, depth=3, seed=42)
print(rc.draw())
print(f"Qubits: {rc.qubit_count()}, Depth: {rc.depth()}, Gates: {rc.gate_count()}")

## Controlled and Adjoint Modifiers

Beyond the builders, the `Circuit` class supports `controlled()` and `adjoint()`
modifiers for constructing more advanced circuits.

In [ ]:
from qdk_pythonic import Circuit

circ = Circuit()
q = circ.allocate(3)

# Controlled-X (Toffoli-like): q[0] and q[1] control X on q[2]
circ.controlled(circ.x, [q[0], q[1]], q[2])

# Adjoint of S gate (S-dagger)
circ.adjoint(circ.s, q[0])

# Controlled rotation: q[2] controls R1(pi/4) on q[0]
import math
circ.controlled(circ.r1, [q[2]], math.pi / 4, q[0])

print(circ.draw())
print()
print(circ.to_qsharp())

## Composing Builders with Manual Gates

Builder circuits are regular `Circuit` objects. You can continue adding gates
after construction:

In [ ]:
circ = ghz_state(3)
q = circ.qubits

# Add a controlled-Z and measurements on top of the GHZ state
circ.cz(q[0], q[2])
circ.measure_all()

print(circ.draw())
print(f"Total gates: {circ.gate_count()}")

## Raw Q# Escape Hatch

Some constructs cannot be expressed with gate-level methods: repeat-until-success
loops, classical control flow, Q# standard library calls. For these cases, use
`raw_qsharp()` to embed Q# fragments directly into the circuit.

The fragment is inserted verbatim into the generated Q# output. It can be
interleaved with regular gate calls and chained fluently.

> **Note:** Circuits containing `raw_qsharp()` fragments cannot be exported to
> OpenQASM. Analysis methods (`depth()`, `gate_count()`, `draw()`) skip raw
> fragments since their cost cannot be determined statically.

In [ ]:
circ = Circuit()
q = circ.allocate(2)

# Mix gate-level methods with raw Q# fragments
circ.h(q[0])
circ.raw_qsharp("let r = M(q[0]);")
circ.raw_qsharp("if r == One { X(q[1]); }")

print("Q# output:")
print(circ.to_qsharp())

In [ ]:
# Fluent chaining works too
circ2 = Circuit()
q2 = circ2.allocate(3)
circ2.h(q2[0]).raw_qsharp("ApplyToEach(H, q[1..2]);").cx(q2[0], q2[1])

print(circ2.to_qsharp())